# Ice Cream Sales vs. Temperature Analysis

**Goal:** Investigate whether there is a relationship between temperature and ice cream sales.

- **H₀:** There is no relationship between temperature and ice cream sales.  
- **H₁:** There is a relationship between temperature and ice cream sales.

We'll use a **one-way ANOVA** test, grouping sales data by temperature bins and checking whether mean sales differ significantly across groups.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("blaban.csv")
df.head()

## Exploratory Plot: Sales by Temperature Bin (Before Any Adjustments)

Before cleaning or merging bins, let's first look at total ice cream sales across raw 5-degree temperature bins to get a sense of the distribution.

In [ ]:
# Filter for ice cream only
df_ice = df[df['Category'] == 'Ice Cream'].copy()

# Bin temperatures into 5-degree intervals
df_ice['Temperature_Bin'] = (df_ice['Temperature_Celsius'] // 5) * 5

# Total sales per bin
sales_by_bin_raw = df_ice.groupby('Temperature_Bin')['Total_Sales'].sum().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(x='Temperature_Bin', y='Total_Sales', data=sales_by_bin_raw,
            hue='Temperature_Bin', legend=False, palette='viridis')
plt.title('Total Ice Cream Sales by Temperature Bin (Unmerged)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Total Sales')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Data Preparation: Average Daily Sales per Bin

Looking at total sales per bin can be misleading — some temperature ranges simply have more days recorded than others. To make a fair comparison, we calculate **average daily sales** per bin.

Also, the 45°C bin has very few observations, so we merge it into the 40°C bin to avoid unreliable group estimates.

In [ ]:
df_ice_clean = df[df['Category'] == 'Ice Cream'].copy()

# Extract date from datetime
df_ice_clean['Date'] = pd.to_datetime(df_ice_clean['Date_Time']).dt.date

# Create 5-degree bins
df_ice_clean['Temperature_Bin'] = (df_ice_clean['Temperature_Celsius'] // 5) * 5

# Merge the sparse 45°C bin into 40°C
df_ice_clean.loc[df_ice_clean['Temperature_Bin'] == 45, 'Temperature_Bin'] = 40

# Total sales per bin
sales_by_bin = df_ice_clean.groupby('Temperature_Bin')['Total_Sales'].sum().reset_index()

# Count unique days per bin (to normalize)
days_per_bin = df_ice_clean.groupby('Temperature_Bin')['Date'].nunique().reset_index()
days_per_bin.rename(columns={'Date': 'Number_of_Days'}, inplace=True)

# Merge and calculate average daily sales
sales_by_bin = pd.merge(sales_by_bin, days_per_bin, on='Temperature_Bin', how='left')
sales_by_bin['Average_Daily_Sales'] = sales_by_bin['Total_Sales'] / sales_by_bin['Number_of_Days']

plt.figure(figsize=(10, 5))
sns.barplot(x='Temperature_Bin', y='Average_Daily_Sales', data=sales_by_bin,
            hue='Temperature_Bin', legend=False, palette='viridis')
plt.title('Average Daily Ice Cream Sales by Temperature Bin (40 & 45°C Merged)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Average Daily Sales')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Statistical Test: One-Way ANOVA

We use a **one-way ANOVA** to test whether average ice cream sales differ significantly across temperature bins.

- Each temperature bin is treated as a group.
- We compare the variance *between* groups to the variance *within* groups.
- If sales are unrelated to temperature, we'd expect no meaningful difference across bins.

In [ ]:
# Group sales values by temperature bin
temp_bins = df_ice_clean['Temperature_Bin'].unique()
groups = [df_ice_clean['Total_Sales'][df_ice_clean['Temperature_Bin'] == b] for b in temp_bins]

# Run one-way ANOVA
f_stat, p_value = stats.f_oneway(*groups)

print(f"F-statistic: {f_stat:.4f}")
print(f"P-value:     {p_value:.4f}")

## Conclusion

With a p-value of **0.12**, which is greater than our significance level of **0.05**, we **fail to reject the null hypothesis**.

This means the data does not provide sufficient evidence to conclude that temperature has a significant effect on ice cream sales at the 95% confidence level.